In [5]:
import pandas as pd

In [6]:
import numpy as np

In [7]:
df = pd.read_csv('../data/cleaned_data/clean_stock_market.csv')

In [8]:
df.shape

(18557, 34)

In [5]:
# Which stocks show the strongest momentum right now (price above SMA-20, RSI in the 50-70 zone)?

In [9]:
df['last_trade_date']=pd.to_datetime(df['last_trade_date'])

In [10]:
valid_df = df.dropna(subset=['last_trade_date']).copy()

latest_date = valid_df['last_trade_date'].max()

latest_df = valid_df[valid_df['last_trade_date'] == latest_date].copy()

momentum_df = latest_df.loc[
    (latest_df['close_price']>
     latest_df['sma_20']) &
     (latest_df['rsi_14'].between(50,70))
].copy()

momentum_df['momentum_pct'] =(
    (momentum_df['close_price'] -
     momentum_df['sma_20'])
     / momentum_df['sma_20']
) *100

top_10_stocks = (
    momentum_df
    .sort_values('momentum_pct',
                 ascending=False)
                 [['company_id','company_name','last_trade_date','close_price',
                   'sma_20','rsi_14','momentum_pct']]
                   .head(10)
)
print(top_10_stocks.to_string(index=False))

 company_id         company_name last_trade_date  close_price  sma_20  rsi_14  momentum_pct
       5009       Bank of Baroda      2026-09-18       179.53   171.8   69.08      4.499418
       5014 HCL Technologies Ltd      2026-09-18      1770.73  1748.0   52.12      1.300343


In [8]:
# How do Banking vs IT sectors compare on next-day direction (Up/Down/Flat)?

In [11]:
Sector_comparison = (
    df[(df['sector'].isin(['Private Bank','Public Sector Bank','Small Finance Bank','IT Services'])) &(df['next_day_direction'] != 'Unknown')]
    .groupby('sector')['next_day_direction']
    .value_counts(normalize=True)
    .unstack()
    .mul(100)
    .round(2)
)
print(Sector_comparison)

next_day_direction   Down   Flat     Up
sector                                 
IT Services         45.23  10.26  44.51
Private Bank        44.35  10.95  44.70
Public Sector Bank  44.30  11.51  44.19
Small Finance Bank  42.89  10.07  47.04


In [10]:
# Which stocks are trading closest to their 52-week high or low?

In [12]:
latest_date = df['last_trade_date'].max()

latest_df = df[df['last_trade_date'] == latest_date].copy()

closest_to_high = (
    latest_df
    .dropna(subset=['pct_from_52w_high'])
    .sort_values('pct_from_52w_high',ascending=False)
    [['company_name','close_price','week52_high','pct_from_52w_high']]
    .head(5)
)

closest_to_low = (
    latest_df
    .dropna(subset=['pct_from_52w_low'])
    .sort_values('pct_from_52w_low',ascending=True)
    [['company_name','close_price','week52_low','pct_from_52w_low']]
    .head(5)
)

print('Top 5 stocks closest to 52-week high')
display(closest_to_high.round(2))

print('Top 5 stocks closest to 52-week low')
display(closest_to_low.round(2))

Top 5 stocks closest to 52-week high


,company_name,close_price,week52_high,pct_from_52w_high
2803,HCL Technologies Ltd,1770.73,1930.74,-8.29
11772,Mphasis Ltd,3443.48,3809.69,-9.61
6423,Infosys Ltd,3476.32,3972.41,-12.49
4545,L&T Technology Services Ltd,3809.06,4366.18,-12.76
13027,Tech Mahindra Ltd,1448.94,1686.11,-14.07


Top 5 stocks closest to 52-week low


,company_name,close_price,week52_low,pct_from_52w_low
1875,Persistent Systems Ltd,2774.70,2701.37,2.71
11551,State Bank of India,585.43,566.83,3.28
14480,IndusInd Bank Ltd,771.18,714.67,7.91
949,AU Small Finance Bank Ltd,707.19,654.68,8.02
3521,Kotak Mahindra Bank Ltd,1365.10,1229.47,11.03


In [12]:
# Which stocks/companies have the highest trading activity (number of trades, trades per day)?

In [13]:
top_liquid_stocks = (
    df
    .dropna(subset=['company_name','trades_per_day','number_of_trades'])
    .groupby('company_name')[['trades_per_day','number_of_trades']]
    .mean()
    .reset_index()
    .sort_values('trades_per_day',ascending=False)
    .head(5)
)
print('=== Top 5 most liquid stocks ===')
display(top_liquid_stocks.round(2))

=== Top 5 most liquid stocks ===


,company_name,trades_per_day,number_of_trades
1,Axis Bank Ltd,6.39,32995.69
16,Persistent Systems Ltd,6.30,31462.86
20,Tech Mahindra Ltd,6.25,32144.55
12,Kotak Mahindra Bank Ltd,6.22,31739.66
18,State Bank of India,6.19,32826.42


In [14]:
# What does delivery percentage reveal about investment vs speculative activity, by settlement type and sector?


In [14]:
delivery_summary = (
    df
    .dropna(subset=['sector','settlement_type','delivery_pct'])
    .groupby(['settlement_type','sector'])
    .agg(
        avg_delivery_pct = ('delivery_pct','mean'),
        total_records = ('delivery_pct','count')
    )
    .reset_index()
    .sort_values(by='avg_delivery_pct', ascending=False)
)
print('=== sector & settlement type wise delivery analysis ===')
display(delivery_summary.round(2))

=== sector & settlement type wise delivery analysis ===


,settlement_type,sector,avg_delivery_pct,total_records
2,Btst,Public Sector Bank,45.78,199
7,Delivery,Small Finance Bank,45.68,112
10,Intraday,Public Sector Bank,45.53,553
14,T+1,Public Sector Bank,45.26,1066
9,Intraday,Private Bank,45.26,1462
15,T+1,Small Finance Bank,45.14,366
13,T+1,Private Bank,44.79,2882
8,Intraday,IT Services,44.75,1809
12,T+1,IT Services,44.72,3542
5,Delivery,Private Bank,44.63,819


In [16]:
# Which market cap segments carry the highest circuit-hit risk?

In [15]:
valid_df = df[
    (df['market_cap_segment'] != 'Unknown') &
    (df['market_cap_segment'].notna()) &
    (df['circuit_hit'].notna())
]


circuit_risk = (
    valid_df
    .groupby('market_cap_segment')['circuit_hit']
    .mean()
)

circuit_risk = (circuit_risk *100) .round(2)
circuit_risk = circuit_risk.sort_values(ascending=False)

print('Market cap segment circuit risk')
display(circuit_risk)

Market cap segment circuit risk


market_cap_segment
Large Cap    2.11
Mid Cap      1.61
Name: circuit_hit, dtype: object

In [18]:
# Is trading status (Active/Suspended/Halted) linked to volatility (ATR) or sector?

In [20]:
df_temp = df.copy()
df_temp['trading_status'] = df_temp['trading_status'].replace('Unknown','Unclassified')

atr_status_analysis = (
    df_temp
    .dropna(subset=['trading_status'])
    .groupby('trading_status')
    .agg(
        total_rows=('atr_14','size'),
        available_atr_rows=('atr_14','count'),
        avg_atr =('atr_14','mean'),
        median_atr =('atr_14','median')
    )
    .reset_index()
)

atr_status_analysis['ATR_availability_pct'] = (
    atr_status_analysis['available_atr_rows']
    / atr_status_analysis['total_rows'] *100
).round(2)

atr_status_analysis[['avg_atr','median_atr']] =(
    atr_status_analysis[['avg_atr','median_atr']].round(2)
)

atr_status_analysis =atr_status_analysis.sort_values(
    by='avg_atr',ascending=False
)
print('TRADING STATUS VS ATR VOLATILITY')
display(atr_status_analysis)

TRADING STATUS VS ATR VOLATILITY


,trading_status,total_rows,available_atr_rows,avg_atr,median_atr,ATR_availability_pct
0,Active,5908,5609,35.94,14.56,94.94
4,Unclassified,10133,9650,35.67,14.54,95.23
3,Suspended,853,806,34.47,13.90,94.49
1,Available,817,779,32.37,11.90,95.35
2,Halted,846,817,31.77,13.05,96.57


Recheck

In [20]:
# How concentrated is open interest across settlement modes (Cash/F&O;/Margin)?

In [18]:
df_tempr = df.copy()
df_tempr['settlement_mode'] = df_tempr['settlement_mode'].replace('Unknown','Unclassified')

oi_concentration = (
    df_tempr
    .groupby('settlement_mode',as_index=False)
    .agg(total_open_interest =('open_interest_lots','sum'))
)

total_oi = oi_concentration['total_open_interest'].sum()

oi_concentration['oi_share_pct'] = (
    oi_concentration['total_open_interest'] / total_oi *100).round(2)

oi_concentration = oi_concentration.sort_values(
    by='oi_share_pct',
    ascending=False
)

hhi_score = (oi_concentration['oi_share_pct']**2).sum()
print('open interest concentration by settlement mode')
display(oi_concentration)



open interest concentration by settlement mode


,settlement_mode,total_open_interest,oi_share_pct
0,Cash,294526460.0,56.90
2,Margin,119476806.0,23.08
1,F&O,71545532.0,13.82
3,Unclassified,32029187.0,6.19


In [22]:
# How complete/reliable is each data source (Moneycontrol/Bloomberg/Reuters/NSE Bhavcopy)?

In [21]:
def audit_data_completeness(df,source_name='Dataset'):
    total_rows = len(df)
    total_cols = len(df.columns)
    print(f'=== data completeness audit:{source_name} ===')
    print(f'total rows:{total_rows:,}')
    print(f'total columns:{total_cols}')

    audit_df = pd.DataFrame({
        'total_records': total_rows,
        'valid_records' : df.notna().sum(),
        'missing_records': df.isna().sum(),
        'data_type' : df.dtypes.astype(str)
    })

    audit_df['completeness_pct'] = (
        audit_df['valid_records'] /
        total_rows *100
    ) .round(2)

    audit_df = audit_df.sort_values(
        by='completeness_pct',
        ascending=True
    )
    avg_completeness = audit_df['completeness_pct'].mean()

    print(f'\nAverage completeness:{avg_completeness:2f}%')

    if avg_completeness >= 95:
        print('completeness status : HIGH')
    elif avg_completeness >= 80:
        print('completeness status: MODERATE')
    else:
        print('completeness status:LOW')
    return audit_df
audit_df = audit_data_completeness(df,source_name='stock market dataset')
display(audit_df)

=== data completeness audit:stock market dataset ===
total rows:18,557
total columns:34

Average completeness:98.237941%
completeness status : HIGH


,total_records,valid_records,missing_records,data_type,completeness_pct
delivery_pct,18557,15758,2799,float64,84.92
trades_per_day,18557,16729,1828,float64,90.15
analyst_rating,18557,16747,1810,float64,90.25
last_trade_date,18557,17016,1541,datetime64[us],91.70
sma_20,18557,17021,1536,float64,91.72
atr_14,18557,17661,896,float64,95.17
circuit_hit,18557,17848,709,object,96.18
trade_id,18557,18557,0,int64,100.00
close_price,18557,18557,0,float64,100.00
market_cap_segment,18557,18557,0,str,100.00


In [24]:
# Does a higher analyst rating actually predict an "Up" next-day direction?

In [22]:
df_clean = df.copy()

df_clean = df_clean.dropna(
    subset=['analyst_rating']
)
df_clean['direction_clean'] =(
    df_clean['next_day_direction']
    .astype(str)
    .str.strip()
    .str.lower()
)
valid_directions = ['up','down','flat']

df_clean = df_clean[
    df_clean['direction_clean'].isin(valid_directions)
]

df_clean['is_up'] =(
    df_clean['direction_clean'] == 'up'
).astype(int)

df_clean['rating_tier'] = pd.cut(
    df_clean['analyst_rating'],
    bins=[0,1,2,3,4,5],
    labels=[
        '0-1 (very low)',
        '1-2 (low)',
        '2-3 (moderate)',
        '3-4 (high)',
        '4-5 (very high)'
    ]
)

prediction_analysis =(
    df_clean
    .groupby('rating_tier', observed=False)
    .agg(
        total_stocks=('is_up','count'),
        up_stocks=('is_up','sum')
    )
    .reset_index()
)
prediction_analysis['up_probability_pct'] = (
    (
        prediction_analysis['up_stocks'].astype(float) / prediction_analysis['total_stocks'].astype(float)
    ) *100
).fillna(0).round(2)

display(prediction_analysis)

,rating_tier,total_stocks,up_stocks,up_probability_pct
0,0-1 (very low),0,0,0.00
1,1-2 (low),5,3,60.00
2,2-3 (moderate),946,402,42.49
3,3-4 (high),10151,4545,44.77
4,4-5 (very high),5629,2528,44.91
